# Importações

In [2]:
!pip install optuna imbalanced-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.3 MB/s eta 0:00:00


In [3]:
import os, copy, random, gc, warnings
import numpy as np
import pandas as pd
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

import optuna
# Reduzindo a verbosidade do Optuna para não poluir a saída visual com 240 logs (30 trials x 8 configs)
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

# Reprodutibilidade

In [ ]:
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device.type.upper()}')

# Caminhos

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Defina os diretórios de entrada e saída
BASE_DIR   = ''
OUTPUT_DIR = ''
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dicionário de Configurações

In [6]:
CONFIGS = {
    'bumbabert_final_com_rag': {
        'path'    : os.path.join(BASE_DIR, 'hidden_states_bumbabert_final.parquet'),
        'desc'    : 'BumbaBERT | Final (28ª) | COM RAG',
        'pred_col': 'pred_aft_bumbabert_final_com_rag',
    },
    'bumbabert_final_sem_rag': {
        'path'    : os.path.join(BASE_DIR, 'hidden_states_bumbabert_final_sem_rag.parquet'),
        'desc'    : 'BumbaBERT | Final (28ª) | SEM RAG',
        'pred_col': 'pred_aft_bumbabert_final_sem_rag',
    },
    'bumbabert_inter_com_rag': {
        'path'    : os.path.join(BASE_DIR, 'hidden_states_bumbabert_intermediario.parquet'),
        'desc'    : 'BumbaBERT | Intermediária (17ª) | COM RAG',
        'pred_col': 'pred_aft_bumbabert_inter_com_rag',
    },
    'bumbabert_inter_sem_rag': {
        'path'    : os.path.join(BASE_DIR, 'hidden_states_bumbabert_intermediario_sem_rag.parquet'),
        'desc'    : 'BumbaBERT | Intermediária (17ª) | SEM RAG',
        'pred_col': 'pred_aft_bumbabert_inter_sem_rag',
    },
    'headtail_final_com_rag': {
        'path'    : os.path.join(BASE_DIR, 'hidden_states_headtail_final.parquet'),
        'desc'    : 'Head+Tail | Final (28ª) | COM RAG',
        'pred_col': 'pred_aft_headtail_final_com_rag',
    },
    'headtail_final_sem_rag': {
        'path'    : os.path.join(BASE_DIR, 'hidden_states_headtail_final_sem_rag.parquet'),
        'desc'    : 'Head+Tail | Final (28ª) | SEM RAG',
        'pred_col': 'pred_aft_headtail_final_sem_rag',
    },
    'headtail_inter_com_rag': {
        'path'    : os.path.join(BASE_DIR, 'hidden_states_headtail_intermediario.parquet'),
        'desc'    : 'Head+Tail | Intermediária (17ª) | COM RAG',
        'pred_col': 'pred_aft_headtail_inter_com_rag',
    },
    'headtail_inter_sem_rag': {
        'path'    : os.path.join(BASE_DIR, 'hidden_states_headtail_intermediario_sem_rag.parquet'),
        'desc'    : 'Head+Tail | Intermediária (17ª) | SEM RAG',
        'pred_col': 'pred_aft_headtail_inter_sem_rag',
    },
}

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

y_verdadeiro = y_true  
y_predito = pred_aft_headtail_final_com_rag   

labels_irdr = ['IRDR-1', 'IRDR-2', 'IRDR-3', 'IRDR-4', 'IRDR-5', 'IRDR-7', 'IRDR-8']

cm = confusion_matrix(y_verdadeiro, y_predito)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_irdr, yticklabels=labels_irdr,
            cbar=False, annot_kws={"size": 12})

plt.ylabel('Classe Verdadeira', fontsize=12, fontweight='bold')
plt.xlabel('Classe Predita', fontsize=12, fontweight='bold')
plt.title('Matriz de Confusão: AFT (Head+Tail, Camada 28, Com RAG)', fontsize=13, pad=15)

plt.tight_layout()
plt.savefig('matriz_confusao_campeao.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

NameError: name 'y_true' is not defined

# Filtro de Classes Raras

In [ ]:
K_FOLDS               = 5
MIN_AMOSTRAS_POR_FOLD = 3
LIMIAR_MINIMO         = K_FOLDS * MIN_AMOSTRAS_POR_FOLD

In [ ]:
def filtrar_classes_raras(df, coluna_rotulo='tipo', limiar=LIMIAR_MINIMO):
    contagem = df[coluna_rotulo].value_counts()
    removidas = contagem[contagem < limiar].index.tolist()
    df_filtrado = df[~df[coluna_rotulo].isin(removidas)].copy()
    return df_filtrado, removidas

# Arquitetura AFT

In [ ]:
class HiddenStatesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

In [ ]:
class AFTClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_classes),
        )
    def forward(self, x): return self.net(x)

# Motor de Treinamento

In [ ]:
def treinar_kfold(X, y, hp, input_size, num_classes, k_folds=K_FOLDS, seed=SEED, verbose=True):
    """
    Motor central de treinamento. Usado tanto para a otimização rápida (k=2)
    quanto para a avaliação final robusta (k=5).
    """
    kf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)
    predicoes = np.zeros(len(y), dtype=int)
    f1_por_fold = []

    PATIENCE   = 20
    MAX_EPOCHS = 300

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # SMOTE - Aplicado apenas no treino para lidar com o forte desbalanceamento
        k_viz = max(1, min(5, int(np.bincount(y_tr).min()) - 1))
        X_tr_res, y_tr_res = SMOTE(random_state=seed, k_neighbors=k_viz).fit_resample(X_tr, y_tr)

        train_loader = DataLoader(HiddenStatesDataset(X_tr_res, y_tr_res),
                                  batch_size=hp['batch_size'], shuffle=True)
        val_loader   = DataLoader(HiddenStatesDataset(X_val, y_val),
                                  batch_size=256, shuffle=False)

        model = AFTClassifier(input_size, hp['hidden_size'], num_classes, hp['dropout']).to(device)
        criterion = nn.CrossEntropyLoss() # Sem class_weight, pois o SMOTE já equilibrou
        optimizer = optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['weight_decay'])

        best_f1      = -1.0
        patience_cnt = 0
        best_weights = copy.deepcopy(model.state_dict())

        for epoch in range(MAX_EPOCHS):
            model.train()
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                optimizer.zero_grad()
                criterion(model(Xb), yb).backward()
                optimizer.step()

            # Early stopping monitorando o F1-Macro (vital para classes raras)
            model.eval()
            preds_ep = []
            with torch.no_grad():
                for Xb, _ in val_loader:
                    preds_ep.extend(torch.argmax(model(Xb.to(device)), 1).cpu().numpy())

            f1_ep = f1_score(y_val, preds_ep, average='macro', zero_division=0)

            if f1_ep > best_f1:
                best_f1 = f1_ep
                best_weights = copy.deepcopy(model.state_dict())
                patience_cnt = 0
            else:
                patience_cnt += 1
                if patience_cnt >= PATIENCE:
                    break

        # Predições do fold usando o melhor checkpoint
        model.load_state_dict(best_weights)
        model.eval()
        fold_preds = []
        with torch.no_grad():
            for Xb, _ in val_loader:
                fold_preds.extend(torch.argmax(model(Xb.to(device)), 1).cpu().numpy())

        predicoes[val_idx] = fold_preds
        f1_fold = f1_score(y_val, fold_preds, average='macro', zero_division=0)
        f1_por_fold.append(f1_fold)

        if verbose:
            epoca_parada = epoch + 1
            epoca_melhor = epoca_parada - patience_cnt
            print(f'  Fold {fold+1}: F1-Macro={f1_fold:.4f} | melhor época={epoca_melhor} | parou em={epoca_parada}')

        del model, optimizer, criterion
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return f1_por_fold, predicoes

# Loop Principal

In [ ]:
# Para cada configuração, ela acha o HP ideal (2-folds)
# e depois treina a avaliação oficial (5-folds) usando o HP recém-descoberto.

df_resultados_finais = None
relatorios_texto     = []
f1_resumo            = []
data_hora_execucao   = datetime.now().strftime('%Y%m%d_%H%M%S')

N_TRIALS = 100

print("INICIANDO PIPELINE DE OTIMIZAÇÃO E AVALIAÇÃO INDEPENDENTE (AFT)\n")

for nome_config, config in CONFIGS.items():
    print(f"{'='*70}")
    print(f"CENÁRIO: {config['desc']}")
    print(f"{'='*70}")

    # Carregamento e Preparação dos Dados
    df_dados = pd.read_parquet(config['path'])
    df_dados['tipo'] = df_dados['tipo'].astype(str).str.strip()

    df_filtrado, removidas = filtrar_classes_raras(df_dados)
    if removidas:
        print(f'  [AVISO] Classes removidas (< {LIMIAR_MINIMO} amostras): {removidas}')

    le_config = LabelEncoder()
    y = le_config.fit_transform(df_filtrado['tipo'])
    X = np.vstack(df_filtrado['hidden_state'].values)

    input_size = X.shape[1]
    num_classes = len(le_config.classes_)
    dist = dict(zip(le_config.classes_, [int(c) for c in np.bincount(y)]))

    print(f'  Amostras úteis: {len(df_filtrado)} | Dimensão: {input_size}')
    print(f'  Distribuição: {dist}\n')

    # Inicializa o DF de resultados definindo o nrprocesso como ÍNDICE ABSOLUTO
    if df_resultados_finais is None:
        df_resultados_finais = pd.DataFrame({
            'y_true': df_filtrado['tipo'].values
        }, index=df_filtrado['nrprocesso'].values)
        df_resultados_finais.index.name = 'nrprocesso'
        label_encoder = le_config

    # FASE DE OTIMIZAÇÃO (OPTUNA - 2 FOLDS)
    print("  [FASE 1] Buscando Hiperparâmetros Ideais (Optuna - 100 Trials)...")

    def objetivo_optuna(trial):
        hp_teste = {
            'hidden_size' : trial.suggest_categorical('hidden_size', [128, 256, 512]),
            'lr'          : trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'dropout'     : trial.suggest_float('dropout', 0.0, 0.5, step=0.1),
            'weight_decay': trial.suggest_float('weight_decay', 1e-4, 1e-1, log=True),
            'batch_size'  : trial.suggest_categorical('batch_size', [16, 32, 64]),
        }
        # Treino rápido com k_folds=2 apenas para orientar a busca
        f1_folds_opt, _ = treinar_kfold(X, y, hp_teste, input_size, num_classes, k_folds=2, verbose=False)
        return np.mean(f1_folds_opt)

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objetivo_optuna, n_trials=N_TRIALS, show_progress_bar=True)

    hp_vencedor = study.best_params
    print(f"  > Melhor F1-Macro (Validação 2-Fold): {study.best_value:.4f}")
    print(f"  > HP Vencedor: {hp_vencedor}\n")

    # FASE DE AVALIAÇÃO FINAL (5 FOLDS)
    print("  [FASE 2] Treinamento Final Oficial (Stratified 5-Fold)...")
    f1_folds, predicoes_num = treinar_kfold(
        X=X, y=y, hp=hp_vencedor, input_size=input_size, num_classes=num_classes, k_folds=K_FOLDS, verbose=True
    )

    # Agrupamento de Resultados (Sem merge, alinhamento direto por index)
    predicoes_texto = le_config.inverse_transform(predicoes_num)

    # Cria uma Series onde o índice é o nrprocesso
    serie_predicoes = pd.Series(predicoes_texto, index=df_filtrado['nrprocesso'].values)

    # O Pandas automaticamente alinha os RGs (índices), ignorando a ordem das linhas.
    df_resultados_finais[config['pred_col']] = serie_predicoes

    f1_medio = np.mean(f1_folds)
    f1_std   = np.std(f1_folds)

    mask = df_resultados_finais[config['pred_col']].notna()
    relatorio_clf = classification_report(
        df_resultados_finais.loc[mask, 'y_true'],
        df_resultados_finais.loc[mask, config['pred_col']],
        zero_division=0
    )

    resumo = (
        f"CENÁRIO       : {config['desc']}\n"
        f"HP Otimizado  : {hp_vencedor}\n"
        f"Classes usadas: {le_config.classes_.tolist()}\n"
        f"F1-Macro Final: {f1_medio:.4f} ± {f1_std:.4f}\n"
        f"{'-'*60}\n"
        f"{relatorio_clf}\n"
        f"{'='*60}\n\n"
    )
    relatorios_texto.append(resumo)

    # Adicionando o HP na tabela de resumo para o paper
    f1_resumo.append({
        'Configuração'   : config['desc'],
        'F1-Macro Médio' : round(f1_medio, 4),
        'Desvio Padrão'  : round(f1_std, 4),
        'Melhor HP'      : str(hp_vencedor), # Salvando como string para caber na tabela
    })

    print(f'\n  RESULTADO FINAL: F1-Macro = {f1_medio:.4f} ± {f1_std:.4f}\n')

# Salvamento

Tabela Comparativa

In [ ]:
df_resumo = pd.DataFrame(f1_resumo)[['Configuração', 'F1-Macro Médio', 'Desvio Padrão', 'Melhor HP']]
df_resumo = df_resumo.sort_values('F1-Macro Médio', ascending=False).reset_index(drop=True)

print("RANKING FINAL DE DESEMPENHO (NON-LINEAR PROBING):")
print(df_resumo.to_string(index=False))

Exportação

In [ ]:
caminho_csv = os.path.join(OUTPUT_DIR, f'Predicoes_AFT_{data_hora_execucao}.csv')
df_resultados_finais.reset_index().to_csv(caminho_csv, index=False)

caminho_txt = os.path.join(OUTPUT_DIR, f'Relatorio_AFT_{data_hora_execucao}.txt')
with open(caminho_txt, 'w', encoding='utf-8') as f:
    f.write('EXPERIMENTO: NON-LINEAR PROBING (AFT) — JUREMA-7B\n')
    f.write(f'Data       : {data_hora_execucao}\n')
    f.write(f'Validação  : Stratified {K_FOLDS}-Fold Cross-Validation\n')
    f.write('Método HP  : Optuna TPE (100 trials em 2-Folds por configuração)\n')
    f.write('='*60 + '\n\n')
    for r in relatorios_texto:
        f.write(r)

print(f'\nArtefatos salvos com sucesso:')
print(f'Predições : {caminho_csv}')
print(f'Relatório : {caminho_txt}')